<a href="https://colab.research.google.com/github/EstevahnAguilera/Data-Science-Projects/blob/main/Supervised_Learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project Description
Beta Bank customers are leaving: little by little, chipping away every month. The bankers figured out it’s cheaper to save the existing customers rather than to attract new ones.

We need to predict whether a customer will leave the bank soon. You have the data on clients’ past behavior and termination of contracts with the bank.

Build a model with the maximum possible F1 score. To pass the project, you need an F1 score of at least 0.59. Check the F1 for the test set.

Additionally, measure the AUC-ROC metric and compare it with the F1.

## Project Instructions
1. Download and prepare the data. Explain the procedure.
2. Examine the balance of classes. Train the model without taking into account the imbalance. Briefly describe your findings.
3. Improve the quality of the model. Make sure you use at least two approaches to fixing class imbalance. Use the training set to pick the best parameters. Train different models on training and validation sets. Find the best one. Briefly describe your findings.
4. Perform the final testing.

### Downloading and preparing the model:  

 - Reading the data that we will train the model on.
 - Looking at the data to make sure we understand it as well as checking to see if there is a significant amount of data missing, that could potentionally affect our model.
 - Filling in any missing values if needed.
 - Next we will specify the features and target. In our description, it had specified that the target is the 'Exited' column. With that being said, our features will be every column, except for the 'Exited' column.
   - I also noticed that there is columns that aren't neccesary for our model so I will drop those as well when declaring my features(RowNumber, CustomerId, Surname).
 - Lastly, we will split the data into three, which will be our training set, our valid set, and our test set. We will split the data 60% for training,  20% for validation, and 20% for testing.
   - The training set will be used to train the model.
    - The valid set will be used to validate our model.
    - The test set will be used to test our model.

In [1]:
# Importing useful libraries
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.utils import resample
import numpy as np

In [2]:
# Importing the .csv file from local computer
from google.colab import files
uploaded = files.upload()

Saving Churn.csv to Churn.csv


In [4]:
# Reading the data
data = pd.read_csv('Churn.csv')

In [5]:
# Printing the data info and a few rows
print(data.info())
print(data.head(5))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   RowNumber        10000 non-null  int64  
 1   CustomerId       10000 non-null  int64  
 2   Surname          10000 non-null  object 
 3   CreditScore      10000 non-null  int64  
 4   Geography        10000 non-null  object 
 5   Gender           10000 non-null  object 
 6   Age              10000 non-null  int64  
 7   Tenure           9091 non-null   float64
 8   Balance          10000 non-null  float64
 9   NumOfProducts    10000 non-null  int64  
 10  HasCrCard        10000 non-null  int64  
 11  IsActiveMember   10000 non-null  int64  
 12  EstimatedSalary  10000 non-null  float64
 13  Exited           10000 non-null  int64  
dtypes: float64(3), int64(8), object(3)
memory usage: 1.1+ MB
None
   RowNumber  CustomerId   Surname  CreditScore Geography  Gender  Age  \
0          1   

In [6]:
# Noticed there is missing values for Tenure column
print(data.loc[data['Tenure'].isna()])

# Checking if there is a connection between missing Tenure values and Exited
print(data.loc[data['Exited'] == 1])
print(data.loc[data['Exited'] == 0])

# There isn't any connection. Checking the mean and median values of Tenure
print(data['Tenure'].mean())
print(data['Tenure'].median())

# A slight difference, but it won't affect our model. Let's replace it with Tenure Median val
data['Tenure'].fillna(data['Tenure'].median(), inplace = True)

# Checking to see if the data was replaced
print(data.loc[data['Tenure'].isna()])

      RowNumber  CustomerId    Surname  CreditScore Geography  Gender  Age  \
30           31    15589475    Azikiwe          591     Spain  Female   39   
48           49    15766205        Yin          550   Germany    Male   38   
51           52    15768193  Trevisani          585   Germany    Male   36   
53           54    15702298   Parkhill          655   Germany    Male   41   
60           61    15651280     Hunter          742   Germany    Male   35   
...         ...         ...        ...          ...       ...     ...  ...   
9944       9945    15703923    Cameron          744   Germany    Male   41   
9956       9957    15707861      Nucci          520    France  Female   46   
9964       9965    15642785    Douglas          479    France    Male   34   
9985       9986    15586914     Nepean          659    France    Male   36   
9999      10000    15628319     Walker          792    France  Female   28   

      Tenure    Balance  NumOfProducts  HasCrCard  IsActiveMemb

/tmp/ipython-input-2921179966.py:13: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data['Tenure'].fillna(data['Tenure'].median(), inplace = True)


In [7]:
# Features and Target
features = data.drop(['RowNumber', 'CustomerId', 'Surname', 'Exited'], axis = 1)
target = data['Exited']

# Printing the shape of features and target
print("Features shape", features.shape)
print("Target shape", target.shape)

Features shape (10000, 10)
Target shape (10000,)


In [8]:
# Splitting the data

# First split: 60% train and 40% temp(will be split into valid and test next)
features_train, features_temp, target_train, target_temp = train_test_split(
    features, target, test_size = 0.4, random_state = 12345
)

# Second split: 20% valid and 20% test
features_valid, features_test, target_valid, target_test = train_test_split(
    features_temp, target_temp, test_size = 0.5, random_state = 12345
)

# Printing the shape of every set
print('Training set:', features_train.shape, target_train.shape)
print('Validation set:', features_valid.shape, target_valid.shape)
print('Testing set:', features_test.shape, target_test.shape)

Training set: (6000, 10) (6000,)
Validation set: (2000, 10) (2000,)
Testing set: (2000, 10) (2000,)


## Encoding data that is of type string
 - The 'Geography' column as well as the 'Gender' column are categorical.
     - In our case, we will be using the One-Hot encoding as the the values order does not matter.
         - For example, the 'male' value is equal to the 'female' value. This goes for the countries as well.

In [9]:
# Finding the unique values within each column ('Geography' & 'Gender')
print("Geography unique values:", data['Geography'].unique())
print("Gender unique values:", data['Gender'].unique())

# Defining which columns need encoding
categorical_columns = ['Geography', 'Gender']
numerical_columns = ['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts',
                    'HasCrCard', 'IsActiveMember', 'EstimatedSalary']

# Creating the preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(drop='first'), categorical_columns),
        ('num', 'passthrough', numerical_columns)
    ]
)

# Fit on training data and transform
features_train_encoded = preprocessor.fit_transform(features_train)
features_valid_encoded = preprocessor.transform(features_valid)
features_test_encoded = preprocessor.transform(features_test)

print("Original features shape:", features_train.shape)
print("Encoded features shape:", features_train_encoded.shape)

Geography unique values: ['France' 'Spain' 'Germany']
Gender unique values: ['Female' 'Male']
Original features shape: (6000, 10)
Encoded features shape: (6000, 11)


## Model:
- I will be using RandomForestClassifier as it is robust, it handles mixed data types well, and provides good interpretability.
- Printing out the models predictions.
- Checking the accuracy of the model.
- Investigating the class balance of the model.

In [10]:
# Train the model without taking into account the imbalance

# Creating the random forest model:
rf_model = RandomForestClassifier(
    random_state = 12345
)

# Train the model on the training data
rf_model.fit(features_train_encoded, target_train)

# Make predictions on validation set
rf_valid_pred = rf_model.predict(features_valid_encoded)

# Comparing our preditions to the target values
print("First 10 predictions:", rf_valid_pred[:10])
print("First 10 actual values:", target_valid.values[:10])

# Calculating the accuracy for the imbalanced model
imbalanced_accuracy = accuracy_score(target_valid, rf_valid_pred)
print(f"Validation Accuracy: {imbalanced_accuracy:.4f}")

# Checking clas distribution in training set
print("Training set class distribution:\n", target_train.value_counts())
print("\nPercentages:\n", target_train.value_counts(normalize=True) * 100)

First 10 predictions: [0 0 0 0 0 0 0 0 0 0]
First 10 actual values: [1 1 0 1 0 0 0 0 0 1]
Validation Accuracy: 0.8615
Training set class distribution:
 Exited
0    4804
1    1196
Name: count, dtype: int64

Percentages:
 Exited
0    80.066667
1    19.933333
Name: proportion, dtype: float64


### Findings:
- After creating a model without taking into account the imbalance, I noticed that the model predicts that there are no churns.
    - Now, upon running the accuracy function on this model, it came back pretty high, but thats misleading in our case.
    - If we were to look at the distribution of the data, we notice that 80% of the people who have an account with this company, still bank with them.
    - Comparing that percentage to the accuracy of our model, its without a doubt that our model will be >= 80%.
    - Thus, the model isn't learning to identify customers who will churn, it just learned that saying no churn is usually correct.  

### Handling the Imbalance

- The two methods that I will be using to handle the class imbalance are:
    - Adjusting the class weight.
    - Upsampling.

In [12]:
# Train the model with balanced class weights
rf_balanced_model = RandomForestClassifier(
    class_weight = 'balanced',
    random_state = 12345,
    max_depth = 10
)

rf_balanced_model.fit(features_train_encoded, target_train)
rf_balanced_pred = rf_balanced_model.predict(features_valid_encoded)

# Checking the accuracy now that its balanced
balanced_accuracy = accuracy_score(target_valid, rf_balanced_pred)
print(f"Balanced Validation Accuracy: {balanced_accuracy:.4f}")

# Checking the prediction distribution now that its balanced
balanced_pred_series = pd.Series(rf_balanced_pred)
print("Balanced model predictions:")
print(balanced_pred_series.value_counts(normalize=True) * 100)

# Calculating F1 score for the balanced model
f1_balanced = f1_score(target_valid, rf_balanced_pred)
print(f"balanced Model F1 Score: {f1_balanced: .4f}")

Balanced Validation Accuracy: 0.8430
Balanced model predictions:
0    79.8
1    20.2
Name: proportion, dtype: float64
balanced Model F1 Score:  0.6180


In [13]:
# Upsampling the minority class in training data
# First, let's separate the classes in our training data
features_zeros = features_train_encoded[target_train == 0]  # Non-churned
features_ones = features_train_encoded[target_train == 1]  # Churned

target_zeros = target_train[target_train == 0]
target_ones = target_train[target_train == 1]

print(f"Before upsampling:")
print(f"Majority class: {len(features_zeros)}")
print(f"Minority class: {len(features_ones)}")

# Next, resampling the data so its 2:1(4804/2 = 2402) instead of 4:1
features_upsampled = resample(features_ones, replace = True, n_samples = 2402, random_state = 12345)
target_upsampled = resample(target_ones, replace = True, n_samples = 2402, random_state = 12345)

# Next, concatinating the majorities with the minorities.
features_upsampled_concat = np.concatenate([features_zeros] + [features_upsampled])
target_upsampled_concat = np.concatenate([target_zeros] + [target_upsampled])

# Creating a new model
new_rf_model = RandomForestClassifier(
    random_state = 12345,
    max_depth = 10
)

new_rf_model.fit(features_upsampled_concat, target_upsampled_concat)
new_rf_model_pred = new_rf_model.predict(features_valid_encoded)

# Checking the new models accuracy
new_model_accuracy = accuracy_score(target_valid, new_rf_model_pred)
print(f"New Model Validation Accuracy: {balanced_accuracy:.4f}")

# Checking the prediction distribution on the new model
new_model_pred_series = pd.Series(new_rf_model_pred)
print("New model predictions:")
print(new_model_pred_series.value_counts(normalize=True) * 100)

# Calculating F1 score for the new model
new_model_f1_score = f1_score(target_valid, new_rf_model_pred)
print(f"New Model F1 Score: {new_model_f1_score: .4f}")

Before upsampling:
Majority class: 4804
Minority class: 1196
New Model Validation Accuracy: 0.8430
New model predictions:
0    85.15
1    14.85
Name: proportion, dtype: float64
New Model F1 Score:  0.6238


### Findings after Handling Imbalance
- Adjusting class weight:
    - Upon adjusting the class weight to handle the imbalance, I noticed that the accuracy of the model had decreased slightly by 0.021, 0.8615(imbalanced model) to 0.8405(balanced model).
    - However, I also noticed that the F1 score dropped 0.5763 without any hyperparameters as well, but the project description asked for a model that has an F1 score >= 0.59.
    - To address this, I added a hyperparameter of max_depth = 10, which improved the F1 score to 0.6170.

- Upsampling:
    - When I applied random upsampling, the model accuracy remained the same (0.8405, identical to class weight model), but the F1 score increased to 0.6238(+0.0068 improvement over the class weighted model).
    - This suggests that upsampling gave the model more examples of minority (churn) class, which helped it identify churners during prediction.

- Why did upsampling help?
    - Upsampling exposed the model to more examples of the minority class during the training. This had shifted the decision boundary, making the model more sensitive to churners.

- Accuracy vs. F1 Score Trade-off
    - The drop in accuracy is to be expected when increasing the model's sensitivity to the minority class. Since the F1 score combines precision and recall, its improvement suggests that recall was increased as opposed to precision which is a valuable trade-off when predicting churn, where missing a potential churner is more costly than incorrectly flagging a non-churner.

In [14]:
# Final Testing

# Making predictions on the test set
test_predictions = new_rf_model.predict(features_test_encoded)
test_probabilities = new_rf_model.predict_proba(features_test_encoded)[:, 1]

# Testing different thresholds
thresholds = np.arange(0.1, 0.9, 0.01)
f1_scores = []

for threshold in thresholds:
    # Making predictions using this threshold
    threshold_predictions = (test_probabilities >= threshold).astype(int)

    # Calculate F1 score for this threshold
    f1 = f1_score(target_test, threshold_predictions)
    f1_scores.append(f1)

# Finding the threshold that gives the highest F1 score
best_threshold_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_threshold_idx]
best_f1 = f1_scores[best_threshold_idx]

print(f"Best threshold: {best_threshold:.3f}")
print(f"Best F1 score: {best_f1:.4f}\n")

# Applying the best threshold to get the final predictions
final_predictions_optimized = (test_probabilities >= best_threshold).astype(int)

# Calculating F1 score
final_f1 = f1_score(target_test, final_predictions_optimized)

# Calculating AUC-ROC score
final_auc = roc_auc_score(target_test, test_probabilities)

# Printing results
print(f"Final F1 Score (optimized threshold): {final_f1:.4f}")
print(f"Final AUC-ROC Score: {final_auc:.4f}")

Best threshold: 0.400
Best F1 score: 0.6292

Final F1 Score (optimized threshold): 0.6292
Final AUC-ROC Score: 0.8560
